# CookSnap Data Cleaning & Dataset Generation


## 1. Import Library

In [43]:
import os
import re
import json
import pandas as pd
import numpy as np

## 2. Konfigurasi

In [62]:
RAW_DIR    = "data/raw"          
OUT_DIR    = "data/data_clean"
os.makedirs(OUT_DIR, exist_ok=True)

CATEGORIES = {
    "ayam":    "Daging",
    "ikan":    "Ikan & Seafood",
    "kambing": "Daging",
    "sapi":    "Daging",
    "tahu":    "Tahu & Tempe",
    "telur":   "Telur & Susu",
    "tempe":   "Tahu & Tempe",
    "udang":   "Ikan & Seafood",
}

# Estimasi waktu masak (menit) berdasarkan keyword di steps
COOK_TIME_HINTS = {
    "presto":  60,
    "rendang": 120,
    "gulai":   45,
    "bakar":   30,
    "goreng":  20,
    "tumis":   15,
    "rebus":   30,
    "kukus":   25,
    "semur":   45,
    "soto":    60,
    "sate":    30,
    "default": 25,
}

print(f"Output directory: {OUT_DIR}")
print(f"Kategori: {list(CATEGORIES.keys())}")

Output directory: data/data_clean
Kategori: ['ayam', 'ikan', 'kambing', 'sapi', 'tahu', 'telur', 'tempe', 'udang']


## 3. Helper Functions

### 3.1 Load Raw CSV

In [63]:
def load_raw(category: str) -> pd.DataFrame:
    """Baca CSV dari raw dir."""
    fname = f"dataset-{category}.csv"
    for candidate in [
        os.path.join(RAW_DIR, fname),
        fname,
    ]:
        if os.path.exists(candidate):
            df = pd.read_csv(candidate)
            df["category"]        = category
            df["main_ingredient"] = category
            return df
    raise FileNotFoundError(f"File {fname} tidak ditemukan.")

### 3.2 Text Cleaning

In [64]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Hilangkan emoji, karakter aneh
    text = re.sub(r'[^\w\s\-\,\.\(\)\/\'\"\!\?]', ' ', text)
    # Multiple spaces jadi satu
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 3.2b Clean Title (khusus nama resep)

In [65]:
def clean_title(title: str) -> str:
    if not isinstance(title, str):
        return ""

    text = title.strip()

    # Hapus prefix penomoran di awal judul, contoh:
    text = re.sub(r'^\s*\d+\s*[\.\)\-_:]*\s*', '', text)

    # Hapus isi di dalam tanda kurung (termasuk kurungnya): (...), [...]
    text = re.sub(r'\([^)]*\)?', ' ', text)   # tangani juga kurung buka tanpa kurung tutup
    text = re.sub(r'\[[^\]]*\]?', ' ', text)
    text = text.replace(')', ' ').replace('(', ' ')  # bersihkan sisa kurung yatim

    # Hapus tanda kutip lurus & tanda kutip pintar (smart quotes)
    text = re.sub(r'["\'\u2018\u2019\u201c\u201d]', '', text)

    # Hapus angka yang tersisa di mana saja dalam judul
    text = re.sub(r'\d+', '', text)

    # Hapus emoji & karakter aneh (pakai clean_text), lalu hapus sisa tanda baca
    text = clean_text(text)
    text = re.sub(r'[.,!?\-_/\\:;]', ' ', text)

    # Rapikan spasi berlebih & trim
    text = re.sub(r'\s+', ' ', text).strip()

    return text

### 3.3 Parse Ingredients

In [66]:
def parse_ingredients(raw: str) -> list[dict]:
    """
    Pisah string bahan (delimiter '--') menjadi list dict.
    Coba ekstrak kuantitas, satuan, dan nama bahan.
    """
    if not isinstance(raw, str) or not raw.strip():
        return []

    items = [s.strip() for s in raw.split("--") if s.strip()]
    result = []
    # Pola: angka + satuan + nama
    pattern = re.compile(
        r'^([\d½¼¾⅓⅔\./\s]+)?\s*'
        r'(kg|gr|gram|g\b|ml|liter|l\b|sdm|sdt|bh|buah|siung|'
        r'batang|lembar|butir|ikat|ruas|cm|bungkus|sachet|kaleng|'
        r'potong|genggam|sdt|sdm|gelas|mangkok|ekor|papan)?\s*'
        r'(.+)$',
        re.IGNORECASE
    )

    skip_words = {"bumbu", "bahan", "halus", "pelengkap", "lain", "toping",
                  "untuk", "campuran", "saus", "sambal", "isian"}

    for item in items:
        item_clean = clean_text(item)
        if not item_clean:
            continue
        # Skip header section
        if item_clean.lower().strip(":") in skip_words:
            continue
        if len(item_clean) < 2:
            continue

        m = pattern.match(item_clean)
        if m:
            qty_raw, unit, name = m.group(1), m.group(2), m.group(3)
            qty_str = (qty_raw or "").strip()
            qty = None
            try:
                qty_str = qty_str.replace("½", "0.5").replace("¼", "0.25").replace("¾", "0.75")
                qty = float(eval(qty_str)) if qty_str else None
            except Exception:
                qty = None
            result.append({
                "ingredient_name": name.strip().lower(),
                "quantity":        qty,
                "unit":            (unit or "").lower() if unit else None,
                "raw_text":        item_clean,
            })
        else:
            result.append({
                "ingredient_name": item_clean.lower(),
                "quantity":        None,
                "unit":            None,
                "raw_text":        item_clean,
            })
    return result

### 3.4 Parse Steps

In [67]:
def parse_steps(raw: str) -> list[str]:
    """Pisah string steps (delimiter '--') menjadi list step bersih."""
    if not isinstance(raw, str) or not raw.strip():
        return []
    steps = [clean_text(s) for s in raw.split("--") if clean_text(s)]
    # Filter terlalu pendek / step kosong
    return [s for s in steps if len(s) > 5]

### 3.5 Estimasi Waktu Masak & Tingkat Kesulitan

In [68]:
def estimate_cook_time(title: str, steps_list: list[str]) -> int:
    """Estimasi waktu masak berdasarkan keyword di judul dan steps."""
    combined = (title + " " + " ".join(steps_list)).lower()
    for kw, minutes in COOK_TIME_HINTS.items():
        if kw != "default" and kw in combined:
            return minutes
    # Cari pola menit eksplisit: "45 menit", "1 jam"
    m = re.search(r'(\d+)\s*(menit|min)', combined)
    if m:
        return min(int(m.group(1)), 180)
    m = re.search(r'(\d+)\s*(jam|hour)', combined)
    if m:
        return min(int(m.group(1)) * 60, 240)
    return COOK_TIME_HINTS["default"]


def infer_difficulty(steps_list: list[str], ingredient_count: int) -> str:
    n_steps = len(steps_list)
    if n_steps <= 4 and ingredient_count <= 7:
        return "Mudah"
    elif n_steps <= 7 and ingredient_count <= 12:
        return "Sedang"
    else:
        return "Sulit"

## 4. Main Cleaning Pipeline

In [69]:
def clean_all():
    all_dfs = []
    for cat in CATEGORIES:
        print(f"  Loading {cat}...")
        df = load_raw(cat)
        df.columns = [c.strip() for c in df.columns]
        # Standarisasi kolom
        rename = {"Title": "title", "Ingredients": "ingredients_raw",
                  "Steps": "steps_raw", "Loves": "loves", "URL": "url"}
        df = df.rename(columns=rename)
        all_dfs.append(df)

    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"Total baris mentah: {len(combined)}")

    # Drop duplikat berdasarkan URL (jika ada), lalu title
    combined = combined.drop_duplicates(subset=["url"],   keep="first")
    combined = combined.drop_duplicates(subset=["title"], keep="first")
    print(f"Setelah dedup: {len(combined)}")

    # Drop baris dengan title/ingredients/steps kosong
    combined = combined.dropna(subset=["title", "ingredients_raw", "steps_raw"])
    combined = combined[combined["ingredients_raw"].str.strip() != ""]
    combined = combined[combined["steps_raw"].str.strip()       != ""]
    print(f"Setelah drop NaN: {len(combined)}")

    # Bersihkan title
    combined["title"] = combined["title"].apply(clean_title).str.strip()

    # Generate recipe_id
    combined = combined.reset_index(drop=True)
    combined["recipe_id"] = ["RCP" + str(i).zfill(5) for i in range(len(combined))]

    # Parse ingredients & steps
    print("Parsing ingredients dan steps...")
    combined["ingredients_list"] = combined["ingredients_raw"].apply(parse_ingredients)
    combined["steps_list"]       = combined["steps_raw"].apply(parse_steps)
    combined["ingredient_count"] = combined["ingredients_list"].apply(len)
    combined["step_count"]       = combined["steps_list"].apply(len)

    # Filter: minimal 2 bahan dan 2 langkah
    combined = combined[(combined["ingredient_count"] >= 2) & (combined["step_count"] >= 2)]
    print(f"Setelah filter kualitas: {len(combined)}")

    # Estimasi waktu & difficulty
    combined["cook_time_min"] = combined.apply(
        lambda r: estimate_cook_time(r["title"], r["steps_list"]), axis=1
    )
    combined["difficulty"] = combined.apply(
        lambda r: infer_difficulty(r["steps_list"], r["ingredient_count"]), axis=1
    )

    # Category label
    combined["category"] = combined["category"].map(CATEGORIES).fillna("Lainnya")

    # Loves (isi 0 jika NaN)
    combined["loves"] = pd.to_numeric(combined["loves"], errors="coerce").fillna(0).astype(int)

    # Estimasi porsi (sederhana)
    combined["servings"] = 2 

    return combined

## 5. Fungsi Save Dataset

In [70]:
def save_recipes(combined: pd.DataFrame):
    recipes = combined[[
        "recipe_id", "title", "category", "main_ingredient",
        "cook_time_min", "difficulty", "servings", "loves",
        "ingredient_count", "step_count", "url"
    ]].copy()
    recipes.to_csv(os.path.join(OUT_DIR, "recipes.csv"), index=False)
    print(f"recipes.csv  ({len(recipes)} baris)")
    return recipes


def save_ingredients(combined: pd.DataFrame):
    rows = []
    for _, row in combined.iterrows():
        for i, ing in enumerate(row["ingredients_list"]):
            rows.append({
                "recipe_id":       row["recipe_id"],
                "step_order":      i + 1,
                "ingredient_name": ing["ingredient_name"],
                "quantity":        ing["quantity"],
                "unit":            ing["unit"],
                "raw_text":        ing["raw_text"],
            })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUT_DIR, "ingredients.csv"), index=False)
    print(f"ingredients.csv  ({len(df)} baris)")
    return df


def save_steps(combined: pd.DataFrame):
    rows = []
    for _, row in combined.iterrows():
        for i, step in enumerate(row["steps_list"]):
            rows.append({
                "recipe_id":   row["recipe_id"],
                "step_number": i + 1,
                "description": step,
            })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUT_DIR, "steps.csv"), index=False)
    print(f"steps.csv  ({len(df)} baris)")
    return df

## 6. Data Nutrisi (Sintetis per 100g)

In [71]:
NUTRITION_DATA = {
    # Format: nama_bahan: {kalori, protein_g, lemak_g, karbo_g, serat_g}
    # Bahan protein utama
    "ayam":           {"calories": 165, "protein": 31.0, "fat":  3.6, "carbs":  0.0, "fiber":  0.0},
    "daging ayam":    {"calories": 165, "protein": 31.0, "fat":  3.6, "carbs":  0.0, "fiber":  0.0},
    "ayam kampung":   {"calories": 150, "protein": 28.0, "fat":  3.2, "carbs":  0.0, "fiber":  0.0},
    "daging sapi":    {"calories": 250, "protein": 26.0, "fat": 15.0, "carbs":  0.0, "fiber":  0.0},
    "sapi":           {"calories": 250, "protein": 26.0, "fat": 15.0, "carbs":  0.0, "fiber":  0.0},
    "daging kambing": {"calories": 234, "protein": 27.0, "fat": 13.0, "carbs":  0.0, "fiber":  0.0},
    "kambing":        {"calories": 234, "protein": 27.0, "fat": 13.0, "carbs":  0.0, "fiber":  0.0},
    "ikan":           {"calories": 105, "protein": 22.0, "fat":  1.5, "carbs":  0.0, "fiber":  0.0},
    "gurame":         {"calories":  96, "protein": 19.0, "fat":  1.2, "carbs":  0.0, "fiber":  0.0},
    "ikan kembung":   {"calories": 103, "protein": 21.0, "fat":  1.8, "carbs":  0.0, "fiber":  0.0},
    "ikan mujaer":    {"calories":  97, "protein": 18.5, "fat":  1.4, "carbs":  0.0, "fiber":  0.0},
    "udang":          {"calories":  99, "protein": 24.0, "fat":  0.3, "carbs":  0.2, "fiber":  0.0},
    "udang besar":    {"calories":  99, "protein": 24.0, "fat":  0.3, "carbs":  0.2, "fiber":  0.0},
    "telur":          {"calories": 155, "protein": 13.0, "fat": 11.0, "carbs":  1.1, "fiber":  0.0},
    "telur ayam":     {"calories": 155, "protein": 13.0, "fat": 11.0, "carbs":  1.1, "fiber":  0.0},
    "tahu":           {"calories":  76, "protein":  8.0, "fat":  4.8, "carbs":  1.9, "fiber":  0.3},
    "tahu putih":     {"calories":  76, "protein":  8.0, "fat":  4.8, "carbs":  1.9, "fiber":  0.3},
    "tahu kuning":    {"calories":  80, "protein":  8.5, "fat":  5.0, "carbs":  2.0, "fiber":  0.3},
    "tempe":          {"calories": 193, "protein": 19.0, "fat": 11.0, "carbs":  9.4, "fiber":  4.1},
    # Bumbu & sayuran
    "bawang merah":   {"calories":  40, "protein":  1.1, "fat":  0.1, "carbs":  9.3, "fiber":  1.7},
    "bawang putih":   {"calories": 149, "protein":  6.4, "fat":  0.5, "carbs": 33.1, "fiber":  2.1},
    "bawang bombai":  {"calories":  40, "protein":  1.1, "fat":  0.1, "carbs":  9.3, "fiber":  1.7},
    "cabe merah":     {"calories":  40, "protein":  1.9, "fat":  0.4, "carbs":  8.8, "fiber":  1.5},
    "cabai merah":    {"calories":  40, "protein":  1.9, "fat":  0.4, "carbs":  8.8, "fiber":  1.5},
    "cabe rawit":     {"calories":  40, "protein":  2.0, "fat":  0.4, "carbs":  8.5, "fiber":  1.5},
    "cabai rawit":    {"calories":  40, "protein":  2.0, "fat":  0.4, "carbs":  8.5, "fiber":  1.5},
    "tomat":          {"calories":  18, "protein":  0.9, "fat":  0.2, "carbs":  3.9, "fiber":  1.2},
    "wortel":         {"calories":  41, "protein":  0.9, "fat":  0.2, "carbs":  9.6, "fiber":  2.8},
    "kentang":        {"calories":  77, "protein":  2.0, "fat":  0.1, "carbs": 17.0, "fiber":  2.2},
    "kol":            {"calories":  25, "protein":  1.3, "fat":  0.1, "carbs":  5.8, "fiber":  2.5},
    "kubis":          {"calories":  25, "protein":  1.3, "fat":  0.1, "carbs":  5.8, "fiber":  2.5},
    "bayam":          {"calories":  23, "protein":  2.9, "fat":  0.4, "carbs":  3.6, "fiber":  2.2},
    "kangkung":       {"calories":  19, "protein":  2.6, "fat":  0.3, "carbs":  3.1, "fiber":  2.0},
    "buncis":         {"calories":  31, "protein":  1.8, "fat":  0.1, "carbs":  7.1, "fiber":  2.7},
    "daun bawang":    {"calories":  32, "protein":  1.8, "fat":  0.2, "carbs":  7.3, "fiber":  2.6},
    "seledri":        {"calories":  16, "protein":  0.7, "fat":  0.2, "carbs":  3.0, "fiber":  1.6},
    "kemangi":        {"calories":  23, "protein":  3.2, "fat":  0.6, "carbs":  2.7, "fiber":  1.6},
    "jahe":           {"calories":  80, "protein":  1.8, "fat":  0.8, "carbs": 18.0, "fiber":  2.0},
    "kunyit":         {"calories": 354, "protein":  7.8, "fat":  9.9, "carbs": 65.0, "fiber": 21.0},
    "lengkuas":       {"calories": 121, "protein":  1.7, "fat":  0.6, "carbs": 26.0, "fiber":  2.1},
    "sereh":          {"calories":  99, "protein":  1.8, "fat":  0.5, "carbs": 25.0, "fiber":  1.8},
    "serai":          {"calories":  99, "protein":  1.8, "fat":  0.5, "carbs": 25.0, "fiber":  1.8},
    "kemiri":         {"calories": 654, "protein":  8.0, "fat": 63.5, "carbs": 13.0, "fiber":  2.7},
    "ketumbar":       {"calories": 298, "protein": 12.4, "fat": 17.8, "carbs": 54.9, "fiber": 10.4},
    "santan":         {"calories": 230, "protein":  2.3, "fat": 24.0, "carbs":  5.5, "fiber":  0.0},
    "kecap manis":    {"calories":  79, "protein":  5.8, "fat":  0.0, "carbs": 15.0, "fiber":  0.0},
    "saus tiram":     {"calories":  37, "protein":  1.0, "fat":  0.4, "carbs":  8.2, "fiber":  0.2},
    "tepung terigu":  {"calories": 364, "protein": 10.3, "fat":  1.0, "carbs": 76.3, "fiber":  2.7},
    "minyak goreng":  {"calories": 884, "protein":  0.0, "fat":100.0, "carbs":  0.0, "fiber":  0.0},
    "garam":          {"calories":   0, "protein":  0.0, "fat":  0.0, "carbs":  0.0, "fiber":  0.0},
    "gula":           {"calories": 387, "protein":  0.0, "fat":  0.0, "carbs":100.0, "fiber":  0.0},
    "lada":           {"calories": 255, "protein": 10.4, "fat":  3.3, "carbs": 63.9, "fiber": 26.5},
    "merica":         {"calories": 255, "protein": 10.4, "fat":  3.3, "carbs": 63.9, "fiber": 26.5},
    "jeruk nipis":    {"calories":  30, "protein":  0.7, "fat":  0.2, "carbs": 11.0, "fiber":  2.8},
    "nasi":           {"calories": 130, "protein":  2.7, "fat":  0.3, "carbs": 28.2, "fiber":  0.4},
    "kornet":         {"calories": 217, "protein": 15.0, "fat": 15.0, "carbs":  4.0, "fiber":  0.0},
    "mentega":        {"calories": 717, "protein":  0.9, "fat": 81.1, "carbs":  0.1, "fiber":  0.0},
}

print(f"Total bahan dalam database nutrisi: {len(NUTRITION_DATA)}")

Total bahan dalam database nutrisi: 57


In [72]:
def save_nutrition():
    rows = []
    for name, vals in NUTRITION_DATA.items():
        rows.append({
            "ingredient_name":  name,
            "calories_per_100g": vals["calories"],
            "protein_g":         vals["protein"],
            "fat_g":             vals["fat"],
            "carbs_g":           vals["carbs"],
            "fiber_g":           vals["fiber"],
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUT_DIR, "nutrition.csv"), index=False)
    print(f"nutrition.csv  ({len(df)} baris)")
    return df

## 7. Jalankan Pipeline

### Step 1 — Load & Clean Raw CSVs

In [73]:
print("=" * 50)
print("CookSnap Data Cleaning & Generation")
print("=" * 50)

print("\n[1] Loading & cleaning raw CSVs...")
combined = clean_all()

CookSnap Data Cleaning & Generation

[1] Loading & cleaning raw CSVs...
  Loading ayam...
  Loading ikan...
  Loading kambing...
  Loading sapi...
  Loading tahu...
  Loading telur...
  Loading tempe...
  Loading udang...
Total baris mentah: 15641
Setelah dedup: 12470
Setelah drop NaN: 12467
Parsing ingredients dan steps...
Setelah filter kualitas: 12267


### Step 2 — Simpan Semua Dataset

In [74]:
print("\n[2] Saving datasets...")
recipes_df     = save_recipes(combined)
ingredients_df = save_ingredients(combined)
steps_df       = save_steps(combined)
nutrition_df   = save_nutrition()


[2] Saving datasets...
recipes.csv  (12267 baris)
ingredients.csv  (149580 baris)
steps.csv  (66980 baris)
nutrition.csv  (57 baris)


### Step 3 — Ringkasan Hasil

In [75]:
print("\n[3] Ringkasan:")
print(f"  Total resep bersih : {len(combined)}")
print(f"  Kategori           : {combined['category'].value_counts().to_dict()}")
print(f"  Difficulty         : {combined['difficulty'].value_counts().to_dict()}")
print(f"\nSemua dataset disimpan di folder: {OUT_DIR}/")
print("Done")


[3] Ringkasan:
  Total resep bersih : 12267
  Kategori           : {'Daging': 4582, 'Ikan & Seafood': 3247, 'Tahu & Tempe': 2899, 'Telur & Susu': 1539}
  Difficulty         : {'Sulit': 5901, 'Sedang': 5494, 'Mudah': 872}

Semua dataset disimpan di folder: data/data_clean/
Done


### Preview Dataset

In [76]:
print("recipes.csv")
display(recipes_df.head())

recipes.csv


,recipe_id,title,category,main_ingredient,cook_time_min,difficulty,servings,loves,ingredient_count,step_count,url
0,RCP00000,Ayam Woku Manado,Daging,ayam,20,Sulit,2,1,14,7,/id/resep/4473027-ayam-woku-manado
1,RCP00001,Ayam goreng tulang lunak,Daging,ayam,60,Sedang,2,1,11,5,/id/resep/4471956-ayam-goreng-tulang-lunak
2,RCP00002,Ayam cabai kawin,Daging,ayam,20,Sedang,2,2,10,3,/id/resep/4473057-ayam-cabai-kawin
3,RCP00003,Ayam Geprek,Daging,ayam,20,Mudah,2,10,7,3,/id/resep/4473023-ayam-geprek
4,RCP00004,Minyak Ayam,Daging,ayam,20,Sedang,2,4,5,6,/id/resep/4427438-minyak-ayam


In [77]:
print("ingredients.csv")
display(ingredients_df.head())

ingredients.csv


,recipe_id,step_order,ingredient_name,quantity,unit,raw_text
0,RCP00000,1,ayam kampung (potong 12),1.0,ekor,1 Ekor Ayam Kampung (potong 12)
1,RCP00000,2,jeruk nipis,2.0,buah,2 Buah Jeruk Nipis
2,RCP00000,3,garam,2.0,sdm,2 Sdm Garam
3,RCP00000,4,kunyit,3.0,ruas,3 Ruas Kunyit
4,RCP00000,5,bawang merah,7.0,None,7 Bawang Merah


In [78]:
print("steps.csv")
display(steps_df.head())

steps.csv


,recipe_id,step_number,description
0,RCP00000,1,Cuci bersih ayam dan tiriskan. Lalu peras jeru...
1,RCP00000,2,"Goreng ayam tersebut setengah matang, lalu tir..."
2,RCP00000,3,Haluskan bumbu menggunakan blender. Bawang mer...
3,RCP00000,4,Setelah bumbu di haluskan barulah di tumis. Ja...
4,RCP00000,5,Masukan ayam yang sudah di goreng setengah mat...


In [79]:
print("nutrition.csv")
display(nutrition_df.head(10))

nutrition.csv


,ingredient_name,calories_per_100g,protein_g,fat_g,carbs_g,fiber_g
0,ayam,165,31.0,3.6,0.0,0.0
1,daging ayam,165,31.0,3.6,0.0,0.0
2,ayam kampung,150,28.0,3.2,0.0,0.0
3,daging sapi,250,26.0,15.0,0.0,0.0
4,sapi,250,26.0,15.0,0.0,0.0
5,daging kambing,234,27.0,13.0,0.0,0.0
6,kambing,234,27.0,13.0,0.0,0.0
7,ikan,105,22.0,1.5,0.0,0.0
8,gurame,96,19.0,1.2,0.0,0.0
9,ikan kembung,103,21.0,1.8,0.0,0.0
